# Modelo Preditivo
### Grupo:
* Gabriel Koji Kondo;
* João Vitor Vargas Pereira;
* Rafael Barreto da Cruz;
* Raissa Arruda Casale;
* Ryan Lionel de Souza.

O objetivo deste notebook é prever o próximo título a ser assitido por um usuário da Netflix, baseado em informações sobre o conteúdo consumido pela conta

## 0. Importações

In [49]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import shap
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.multioutput import ClassifierChain
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import classification_report
from sklearn.inspection import permutation_importance

# Configuração para que os gráficos apareçam no notebook
%matplotlib inline

## 1. Carregamento de dados

In [50]:
df_api = pd.read_csv('../dados-transformados/tmdb_completo.csv')
df_viewing = pd.read_csv('../dados-transformados/ViewingActivityCorrigido.csv')

# 1. Separamos a string em Título, Temporada e Episódio.
df_viewing[['Title', 'Temporada', 'Episodio']] = df_viewing['Title'].str.split(': ', n=2, expand=True)

# 2. Limpamos os parênteses APENAS da nova coluna 'Title'
df_viewing["Title"] = df_viewing["Title"].str.replace(r"\(.*?\)", "", regex=True).str.strip()

# 3. Merge com a base do TMDB
df = df_viewing.merge(df_api, left_on="Title", right_on="Tittle_Name", how="left")

## 2. Pré Processamento

In [51]:
# 1. Converter datas e ordenar para criar a "linha do tempo" do usuário
df["Start Time"] = pd.to_datetime(df["Start Time"], errors="coerce")
df = df.sort_values(["Profile Name", "Start Time"])

# 2. Lógica de Sequência (O que ele viu antes e o que vai ver depois)
df["genero_anterior"] = df.groupby("Profile Name")["genres"].shift(1)
df["genero_atual"] = df["genres"]
df["proximo_genero"] = df.groupby("Profile Name")["genres"].shift(-1)

# 3. Remover última linha de cada usuário (pois não sabemos o "próximo")
df = df.dropna(subset=["proximo_genero"])

# 4. Features Temporais
df["dia_semana"] = df["Start Time"].dt.dayofweek
df["hora"] = df["Start Time"].dt.hour

# 5. Transformar Gênero Atual e Anterior em colunas numéricas (Dummies)
dummies_atual = pd.get_dummies(df["genero_atual"], prefix="flag")
dummies_anterior = pd.get_dummies(df["genero_anterior"], prefix="lag") # O que ele viu 1 passo atrás

# 6. Proporção Histórica (O quanto ele gosta de cada gênero no geral)
props = dummies_atual.groupby(df["Profile Name"]).transform("mean").add_prefix("prop_")

# 7. Juntar tudo no DataFrame e remover colunas com nomes duplicados
df = pd.concat([df, dummies_atual, dummies_anterior, props], axis=1)
df = df.loc[:, ~df.columns.duplicated()]

C:\Users\gabrielkondo-ieg\AppData\Local\Temp\ipykernel_1564\91843440.py:2: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df["Start Time"] = pd.to_datetime(df["Start Time"], errors="coerce")


In [52]:
# 1. Prepara o Y (Alvo Multilabel)
# Transformamos a string "Ação, Comédia" em lista ['Ação', 'Comédia']
y_list = df["proximo_genero"].fillna("Desconhecido").str.split(", ")

mlb = MultiLabelBinarizer()
y_bin = mlb.fit_transform(y_list)

print(f"Número de gêneros únicos a prever: {len(mlb.classes_)}")

# 2. Prepara o X (Features)
# Removemos textos, datas e variáveis que podem "vazar" a resposta
colunas_remover = [
    "proximo_genero", "Start Time", "Profile Name", "Title", "Tittle_Name",
    "genres", "genero_atual", "genero_anterior", "director", "director_list"
]

X = df.drop(columns=colunas_remover, errors="ignore")

# Pulo do Gato: Manter apenas números e booleanos!
X = X.select_dtypes(include=['int64', 'float64', 'uint8', 'int32', 'bool']).fillna(0)

Número de gêneros únicos a prever: 26


In [53]:
# 1. Separando em Treino e Teste
X_train, X_test, y_train, y_test = train_test_split(
    X, y_bin, test_size=0.2, random_state=42
)

# 2. Treinando o modelo com Corrente de Classificadores (ideal para Multilabel)
base_rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
model = ClassifierChain(base_rf, order='random', random_state=42)

print("Treinando o modelo... (Isso pode levar alguns segundos)")
model.fit(X_train, y_train)
print("Treinamento concluído!")

Treinando o modelo... (Isso pode levar alguns segundos)
Treinamento concluído!


## 3. Separação entre treino e teste

## 3.1 Separando os x e y

In [54]:
y_pred = model.predict(X_test)

print("--- Relatório de Classificação Multilabel ---")
print(classification_report(y_test, y_pred, target_names=mlb.classes_, zero_division=0))

--- Relatório de Classificação Multilabel ---
                    precision    recall  f1-score   support

Action & Adventure       0.90      0.24      0.38       373
          Animação       0.66      0.27      0.38       329
          Aventura       1.00      0.01      0.02       115
              Ação       0.87      0.15      0.26        85
         Cinema TV       0.00      0.00      0.00         1
           Comédia       0.79      0.33      0.47       774
             Crime       0.87      0.19      0.31       296
      Documentário       0.00      0.00      0.00        32
             Drama       0.74      0.89      0.81      1217
           Família       0.79      0.13      0.22       239
          Fantasia       1.00      0.03      0.05        37
          Faroeste       0.67      0.11      0.18        19
 Ficção científica       0.88      0.25      0.39        56
            Guerra       0.00      0.00      0.00         6
          História       0.00      0.00      0.00    

In [ ]:
from sklearn.inspection import permutation_importance
import pandas as pd
import matplotlib.pyplot as plt

print("Calculando Permutation Importance (versão otimizada)...")

# Como os dados já foram embaralhados no train_test_split, 
# pegamos as 1000 primeiras linhas com segurança (iloc corta pela posição, evitando erros de índice)
X_test_amostra = X_test.iloc[:1000]
y_test_amostra = y_test[:1000]

# 1. Calculando a Permutation Importance com a amostra
result = permutation_importance(
    model, 
    X_test_amostra, 
    y_test_amostra, 
    n_repeats=3,      # 3 repetições são perfeitas para ter desvio padrão sem demorar
    random_state=42, 
    n_jobs=-1         # Usa todos os núcleos do processador
)

# 2. Criando o DataFrame já com a Média, o Desvio Padrão e ordenado
df_perm_importance = pd.DataFrame({
    'Feature': X_test.columns,
    'Importância': result.importances_mean,
    'Desvio_Padrao': result.importances_std
}).sort_values(by='Importância', ascending=True)

# 3. Filtrando as 15 variáveis mais importantes
top_features = df_perm_importance.tail(15)

# 4. Plotando o Gráfico Otimizado
plt.figure(figsize=(10, 8))

# Criando as barras com a linha de erro (desvio padrão)
plt.barh(
    top_features['Feature'], 
    top_features['Importância'], 
    xerr=top_features['Desvio_Padrao'], 
    color='coral', 
    edgecolor='black',
    capsize=5, 
    alpha=0.9
)

plt.xlabel('Queda na Acurácia (Importância Média)')
plt.title('Top 15 Variáveis Mais Importantes (Permutation Importance)')

# Adicionando uma grade no eixo X para facilitar a leitura dos valores
plt.grid(axis='x', linestyle='--', alpha=0.7) 

plt.tight_layout()
plt.show()

Calculando Permutation Importance (versão otimizada)...


In [ ]:
# Pegamos apenas uma amostra do teste para o SHAP rodar rápido (100 linhas)
X_amostra = X_test.iloc[:100, :]

# Criando o explainer para o primeiro modelo da cadeia
explainer = shap.TreeExplainer(model.estimators_[0])
shap_values = explainer.shap_values(X_amostra)

# Gráfico de resumo SHAP (O que puxa a previsão para cima/baixo)
plt.title(f"Impacto das Features na Previsão do Gênero: {mlb.classes_[0]}")
shap.summary_plot(shap_values, X_amostra, plot_type="bar")

--- Relatório de Classificação Multilabel ---
                    precision    recall  f1-score   support

Action & Adventure       0.82      0.26      0.40       317
          Animação       0.78      0.37      0.50       222
          Aventura       0.00      0.00      0.00        70
              Ação       0.64      0.14      0.23        64
         Cinema TV       0.00      0.00      0.00         2
           Comédia       0.83      0.44      0.58       605
             Crime       0.80      0.28      0.42       234
      Documentário       1.00      0.06      0.11        17
             Drama       0.77      0.93      0.84       992
           Família       0.81      0.23      0.36       164
          Fantasia       1.00      0.05      0.09        21
          Faroeste       1.00      0.36      0.53        14
 Ficção científica       0.67      0.28      0.39        29
            Guerra       0.00      0.00      0.00         2
          História       0.00      0.00      0.00    

In [ ]:
# 1. Pegamos a última atividade registrada de cada usuário
ultimas_atividades = X.loc[df.groupby("Profile Name")["Start Time"].idxmax()]

# 2. Fazemos a previsão Multilabel
preds_binarias = model.predict(ultimas_atividades)

# 3. Transformamos os 1s e 0s de volta em nomes de gêneros
generos_previstos = mlb.inverse_transform(preds_binarias)

# 4. Criamos um DataFrame de resumo limpo
relatorio_usuarios = pd.DataFrame({
    'Usuario': df.groupby("Profile Name")["Profile Name"].first().values,
    'Ultimo_Genero_Visto': df.groupby("Profile Name")["genres"].last().values,
    'Previsao_Proxima_Sessao': [", ".join(g) if g else "Indefinido" for g in generos_previstos]
})

print("--- Sugestões para o Próximo Acesso ---")
display(relatorio_usuarios)

# Descomente a linha abaixo caso queira exportar:
# relatorio_usuarios.to_csv('recomendacoes_multilabel.csv', index=False)

,Usuario,Ultimo_Genero_Visto,Previsao_Proximo
0,Extra,Drama,Drama
1,João Vitor,"Animação, Action & Adventure, Drama, Sci-Fi & ...","Animação, Comédia"
2,Rafael,"Comédia, Drama","Comédia, Drama"
3,Raiss,"Sci-Fi & Fantasy, Drama, Mistério",Drama
4,Ryan,Drama,Drama
